In [7]:
# @title Configurations

from torchvision import transforms
import torch

# --- CONFIGURATIONS: SET PATHS, DEVICE, MODEL AND HYPERPARAMETERS ---

CONFIGURATION = {
	"PATH_DRIVE": "/content/drive",
	"PATH_FILE": "/content/drive/MyDrive/AML-Semantic-Correspondence/datasets/SPair-71k.tar.gz",
  "PATH_BEST_MODEL": "/content/drive/MyDrive/AML-Semantic-Correspondence/weights/best_model.pth",

	"PTH_PATH_DINOV2": "/content/drive/MyDrive/AML-Semantic-Correspondence/weights/dinov2_vitb14_reg4_pretrain.pth",
  "PTH_PATH_DINOV3": "/content/drive/MyDrive/AML-Semantic-Correspondence/weights/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth",
  "PTH_PATH_SAM": "/content/drive/MyDrive/AML-Semantic-Correspondence/weights/sam_vit_b_01ec64.pth",

  "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
  "MODEL_VERSION": "dinov2",                                        # "dinov2", "dinov3", "sam"

  # DATASET SPAIR-71k

  "PATH_TEST_SPAIR71K": "/content/SPair-71k/PairAnnotation/test",
  "PATH_VAL_SPAIR71K": "/content/SPair-71k/PairAnnotation/val",
  "PATH_TRAIN_SPAIR71K": "/content/SPair-71k/PairAnnotation/trn",

  "ALL_TEST_PATH_SPAIR71K": "/content/SPair-71k/Layout/small/test.txt",
  "ALL_TRAIN_PATH_SPAIR71K": "/content/SPair-71k/Layout/small/trn.txt",
  "ALL_VAL_PATH_SPAIR71K": "/content/SPair-71k/Layout/small/val.txt",

  "IMAGE_FOLDER_NAME_SPAIR71K": "/content/SPair-71k/JPEGImages",

  # FOR INFERENCE

	"ALPHA": [0.05, 0.1, 0.2],

  # FOR TUNING

  "TAU": 0.05,
  "LEARNING_RATE": 1e-4,
  "WEIGHT_DECAY": 1e-2,
  "NUM_EPOCHS": 1,
  "NUM_LAYERS": 1,
  "BATCH_SIZE": 16,
  "TUNING": True
}

MODEL = None

if CONFIGURATION["MODEL_VERSION"] == "dinov2":              # DINOV2 -> DIM PATCH -> 14 -> IMAGE SIZE 518
  IMAGE_SIZE = 518
elif CONFIGURATION["MODEL_VERSION"] == "dinov3":            # DINOV3 -> DIM PATCH -> 16 -> IMAGE SIZE 512
  IMAGE_SIZE = 512
else:
  IMAGE_SIZE = 1024                                         # SAM -> DIM PATCH -> 16 -> IMAGE SIZE 1024

# --- RESIZE IMAGE TO STANDARD MODEL DIMENSIONS, CONVERT IT INTO A TENSOR AND NORMALIZE IT ---

PREPROCESS = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [8]:
# @title Datasets

from torch.utils.data import Dataset, DataLoader
# from Configurations import CONFIGURATION
import os, json, numpy as np

# --- DATASET CLASSES ---
# --- COLLECT ALL JSON FILE ---
# --- WHEN REQUIRED, OPEN THE NEXT FILE AND TAKE IN A DICTIONARY ALL YOU NEED ---

class SPair71kDataset(Dataset):

    def __init__(self, pair_path, source_path):
        self.image_path = source_path
        file = open(pair_path, "r")
        self.pair_files = file.readlines()                # TAKE ALL JSON FILES
        file.close()
        return

    def __len__(self):
        return len(self.pair_files)

    def __getitem__(self, idx):
        file_name = self.pair_files[idx].strip()
        category = file_name.split(".json")[0].split(":")[1]
        json_path = os.path.join(self.image_path, file_name + ".json")

        file = open(json_path, "r")
        annotation = json.load(file)             # LOAD INFORMATIONS
        file.close()

        src_path = os.path.join(CONFIGURATION["IMAGE_FOLDER_NAME_SPAIR71K"], category, annotation["src_imname"])
        trg_path = os.path.join(CONFIGURATION["IMAGE_FOLDER_NAME_SPAIR71K"], category, annotation["trg_imname"])
        ids = [int(el) for el in annotation["kps_ids"]]

        return {
            "src_path": src_path,
            "trg_path": trg_path,
            "src_kps": np.array(annotation["src_kps"]),
            "trg_kps": np.array(annotation["trg_kps"]),
            "kps_ids": np.array(ids),
            "trg_bndbox": np.array(annotation["trg_bndbox"]),        # DICT of BATCHES (SIZE=1)
        }

def custom_collate_fn(batch):
    collated_batch = {}
    keys = batch[0].keys()
    for key in keys:
        collated_batch[key] = [d[key] for d in batch]              # COLLATE FOR DATALOADER
    return collated_batch

# --- GENERATE DATALOADER FROM REQUIRED OPERATION --

def Loader(index):            # TEST, TRAINING OR EVALUATION
    if index == 0:
        dataset = SPair71kDataset(CONFIGURATION["ALL_TEST_PATH_SPAIR71K"], CONFIGURATION["PATH_TEST_SPAIR71K"])
        loader = DataLoader(dataset, 1)

    elif index == 1:
        dataset = SPair71kDataset(CONFIGURATION["ALL_TRAIN_PATH_SPAIR71K"], CONFIGURATION["PATH_TRAIN_SPAIR71K"])
        loader = DataLoader(dataset, CONFIGURATION["BATCH_SIZE"], collate_fn=custom_collate_fn)

    elif index == 2:
        dataset = SPair71kDataset(CONFIGURATION["ALL_VAL_PATH_SPAIR71K"], CONFIGURATION["PATH_VAL_SPAIR71K"])
        loader = DataLoader(dataset, 1)

    return loader

In [ ]:
# @title Utils

# from Configurations import CONFIGURATION, PREPROCESS, MODEL
from PIL import Image
import torch, torch.nn.functional as F, math

# --- HELPER FUNCTIONS ---
# --- LOAD AND PREPROCESS IMAGE ---
# --- RESHAPE AND NORMALIZE FEATURE MAPS ---
# --- FINALLY RETURN THE EXTRACTED FEATURE MAPS AND THE ORIGINAL DIMENSIONS ---

def get_descriptors(img_path, grad):
    img = Image.open(img_path).convert("RGB")
    (w, h) = img.size
    input_tensor = PREPROCESS(img).unsqueeze(0).to(CONFIGURATION["DEVICE"])

    with torch.set_grad_enabled(grad):
        if CONFIGURATION["MODEL_VERSION"] == "dinov2":
            x = MODEL.get_intermediate_layers(input_tensor, n=1)[0]   # [B, N, D]
            (B, N, D) = x.shape
            H = int(math.sqrt(N))
            x = x.reshape(B, H, H, D)                # RESHAPE

        elif CONFIGURATION["MODEL_VERSION"] == "dinov3":
            x = MODEL.forward_features(input_tensor)["x_norm_patchtokens"]
            (B, N, D) = x.shape
            H = int(math.sqrt(N))
            x = x.reshape(B, H, H, D)               # RESHAPE

        elif CONFIGURATION["MODEL_VERSION"] == "sam":
            x = MODEL.image_encoder(input_tensor).permute(0, 2, 3, 1)

    x = F.normalize(x, dim=-1)
    return (x, w, h)

# --- GET PREDICTIONS ---
# --- EXTRACT DESCRIPTORS, FOR EACH SRC_KPS RESCALE IT ---
# --- USE COSINE SIMILARITY METRIC AND COMPUTE THE PREDICTION WITH SIMPLE ARGMAX  ---

def get_predictions(batch, index=0):
    src_kps = batch["src_kps"][index]

    (feat_src, sw, sh) = get_descriptors(batch["src_path"][index], False)
    (feat_trg, tw, th) = get_descriptors(batch["trg_path"][index], False)             # DESCRIPTORS

    (_, Hf, Wf, D) = feat_trg.shape
    trg_flat = feat_trg[0].reshape(Hf * Wf, D)

    pred_kps = []

    for i in range(src_kps.shape[0]):
        sx = int(src_kps[i, 0] * Wf / sw)
        sy = int(src_kps[i, 1] * Hf / sh)
        sx = torch.clamp(torch.tensor(sx, device=CONFIGURATION["DEVICE"]), 0, Wf - 1)              # RESIZE
        sy = torch.clamp(torch.tensor(sy, device=CONFIGURATION["DEVICE"]), 0, Hf - 1)

        src_desc = feat_src[0, sy, sx, :]
        sim = torch.matmul(trg_flat, src_desc)            # COSINE SIMILARITY

        best_idx = sim.argmax()
        pred_xy = torch.tensor([best_idx % Wf, best_idx // Wf], device=CONFIGURATION["DEVICE"])           # PREDICTION

        pred_x = (pred_xy[0] + 0.5) * (tw / Wf)
        pred_y = (pred_xy[1] + 0.5) * (th / Hf)
        pred_kps.append(torch.stack([pred_x, pred_y]))

    return torch.stack(pred_kps)

In [10]:
#@title Train

# from Configurations import CONFIGURATION, Loader, MODEL
# from Utils import get_descriptors
# from Inference import run_evaluation
from tqdm.auto import tqdm
import torch, torch.nn.functional as F

# --- UNFREEZE ONLY THE LAST num_last_blocks LAYERS AND THE FINAL LAYER NORM ---
# --- FREEZE ALL LAYERS AND THEN FREE THE LASTS (IF ACCESSIBLE) ---

def setup_light_finetuning():
    N_Params = 0
    N_Free_Params = 0

    if CONFIGURATION["MODEL_VERSION"] == "sam":
        blocks_to_unfreeze = MODEL.image_encoder.blocks[-CONFIGURATION["NUM_LAYERS"]:]

        if hasattr(MODEL.image_encoder, "post_norm"):       # NOT SURE THE FINAL NORM IS ACCESSIBLE
            norm = MODEL.image_encoder.post_norm
        else:
            norm = None

    else:
        blocks_to_unfreeze = MODEL.blocks[-CONFIGURATION["NUM_LAYERS"]:]
        norm = MODEL.norm

    for param in MODEL.parameters():                # FREEZE ALL
        N_Params += param.numel()
        param.requires_grad = False

    for block in blocks_to_unfreeze:

        for param in block.parameters():
            N_Free_Params += param.numel()           # UNFREEZE
            param.requires_grad = True

    if norm:
        for param in norm.parameters():          # UNFREEZE NORM
            N_Free_Params += param.numel()
            param.requires_grad = True

    # NUMBERS

    print("Total parameters:", N_Params)
    print("Total trainable:", N_Free_Params)
    print("Percentage trainable:", round(100 * N_Free_Params / N_Params, 2), "%")
    return

# --- EXTRACT FEATURE SIZE, FOR EACH SRC_KPS RESCALE IT, USE COSINE SIMILARITY METRIC ---
# --- AND COMPUTE THE PREDICTION WITH SIMPLE ARGMAX ---

def get_split_loss(batch, grad=True, index=0):
    src_kps = torch.as_tensor(batch["src_kps"][index]).to(device=CONFIGURATION["DEVICE"], dtype=torch.float32)
    trg_kps = torch.as_tensor(batch["trg_kps"][index]).to(device=CONFIGURATION["DEVICE"], dtype=torch.float32)

    (feat_src, sw, sh) = get_descriptors(batch["src_path"][index], grad)
    (feat_trg, tw, th) = get_descriptors(batch["trg_path"][index], grad)            # GET DESCRIPTORS

    (_, Hf, Wf, D) = feat_trg.shape
    trg_flat = feat_trg[0].reshape(Hf * Wf, D)
    class_loss = 0

    for i in range(src_kps.shape[0]):
        sx = int(src_kps[i, 0] * Wf / sw)
        sy = int(src_kps[i, 1] * Hf / sh)
        sx = torch.clamp(torch.tensor(sx, device=CONFIGURATION["DEVICE"]), 0, Wf - 1)       # FOR EACH KEYPOINT
        sy = torch.clamp(torch.tensor(sy, device=CONFIGURATION["DEVICE"]), 0, Hf - 1)

        src_desc = feat_src[0, sy, sx, :]
        sim = torch.matmul(trg_flat, src_desc)            # COSINE SIMILARITY

        # --- GROUND TRUTH TARGET POSITION RESCALED TO FEATURE MAP ---

        gx = int(trg_kps[i, 0] * Wf / tw)
        gy = int(trg_kps[i, 1] * Hf / th)
        gx = torch.clamp(torch.tensor(gx, device=CONFIGURATION["DEVICE"]), 0, Wf - 1)
        gy = torch.clamp(torch.tensor(gy, device=CONFIGURATION["DEVICE"]), 0, Hf - 1)

        gt_index = gy * Wf + gx
        class_loss = class_loss + F.cross_entropy((sim / CONFIGURATION["TAU"]).unsqueeze(0), gt_index.unsqueeze(0))

    return class_loss

# --- TRAINING FUNCTION ---
# --- FIRST EVALUATE PERFORMANCE WITHOUT FINETUNING, THEN UNFREEZE LAST LAYERS ---
# --- FOR EACH BATCH, COMPUTE CROSS ENTROPY LOSS AND BACK PROPAGATE ---
# --- FINALLY SHOW THE VALUES ---

def Train_step():
    global_loss = float("inf")
    loader_train = Loader(index=1)
    loader_val = Loader(index=2)

    print()
    print("="*60)
    print("PERFORMING INITIAL EVALUATION ON PRE-TRAINED MODEL")
    print("="*60)

    initial_val_loss = get_total_loss(loader_val)
    print()
    run_evaluation(loader_val, "validation")
    print("Pre-tuning Validation Loss: " + str(initial_val_loss))
    print("="*60)
    print()

    print()
    print("Fine-tuning on " + str(CONFIGURATION["NUM_LAYERS"]) + " free layers")
    setup_light_finetuning()
    params = [p for p in MODEL.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=CONFIGURATION["LEARNING_RATE"], weight_decay=CONFIGURATION["WEIGHT_DECAY"])

    for epoch in range(CONFIGURATION["NUM_EPOCHS"]):

        # --- TRAINING PHASE ---

        MODEL.train()
        running_train_loss = 0.0
        pbar = tqdm(loader_train, desc="Epoch " + str(epoch))

        for batch in pbar:
            optimizer.zero_grad()
            bs = len(batch["src_kps"])                # BATCH LOSS TO BACK PROPAGATE
            class_loss = 0.0

            for i in range(bs): class_loss += get_split_loss(batch, index=i)

            class_loss /= bs
            class_loss.backward()
            optimizer.step()
            running_train_loss += class_loss.item()
            pbar.set_postfix(loss=class_loss.item())                    # FINAL EPOCH LOSS

        epoch_train_loss = running_train_loss / len(loader_train)

        # --- VALIDATION PHASE ---

        val_loss = get_total_loss(loader_val)

        print("="*60)
        print("Epoch train loss: " + str(epoch_train_loss))
        print("Epoch validation loss: " + str(val_loss))                  # NUMBERS
        print()
        run_evaluation(loader_val, "validation")

        if val_loss < global_loss:
            global_loss = val_loss
            torch.save(MODEL.state_dict(), CONFIGURATION["PATH_BEST_MODEL"])
            print("BEST MODEL SAVED!")

        print("="*60)
        print()

    return

# --- COMPUTE CROSS ENTROPY LOSS ON EVALUATION DATASET ---

def get_total_loss(loader):
    MODEL.eval()
    total_loss = 0.0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating validation loss"):
            bs = len(batch["src_kps"])
            batch_loss_sum = 0.0

            for i in range(bs):
                class_loss = get_split_loss(batch, grad=False, index=i)           # BATCH LOSS
                batch_loss_sum += class_loss.item()

            total_loss += (batch_loss_sum / bs)          # MEAN

    mean_loss = total_loss / len(loader)           # MEAN
    return mean_loss


In [ ]:
# @title Inference

# from Configurations import CONFIGURATION
# from Utils import get_predictions
from tqdm.auto import tqdm
from matplotlib import pyplot as plt
import torch, cv2

# --- EVALUATION FUNCTION ---
# --- FOR EACH PAIR: LOAD SOURCE AND TARGET IMAGES ---
# --- EXTRACT DESCRIPTORS AND RESCALE COORDINATES ---
# --- USE COSINE SIMILARITY IN ORDER TO FIND THE CORRESPONDING POINT IN THE TARGET IMAGE. ---
# --- FINALLY, RETURN THE CORRECT GENERATED KEYPOINTS (USING THEIR DISTANCE FROM THE ORIGINAL ONE) RATIO. ---
# --- OPTIONALLY: VISUALIZE RESULTS. ---

def run_evaluation(loader, split_desc, visualize=False):
    total_correct = {alpha: 0 for alpha in CONFIGURATION["ALPHA"]}
    total_images = 0
    total_correct_keypoints = {alpha: 0 for alpha in CONFIGURATION["ALPHA"]}
    total_keypoints = 0

    for batch in tqdm(loader, desc="Evaluating " + split_desc + " PCK metrics"):
        trg_kps = torch.as_tensor(batch["trg_kps"][0]).to(device=CONFIGURATION["DEVICE"], dtype=torch.float32)
        trg_bndbox = torch.as_tensor(batch["trg_bndbox"][0]).to(device=CONFIGURATION["DEVICE"], dtype=torch.float32)

        pred_kps = get_predictions(batch)

        max_dim = max(trg_bndbox[2]-trg_bndbox[0], trg_bndbox[3]-trg_bndbox[1])
        total_correct_image = {alpha: 0 for alpha in CONFIGURATION["ALPHA"]}    # KEYPOINTS CORRECTLY CLASSIFIED
        total_points_image = 0                                      # FOR THE CURRENT IMAGE
        total_images += 1

        for i in range(len(batch["kps_ids"][0])):
            total_keypoints += 1

            dist = torch.norm(pred_kps[i] - trg_kps[i]).item()            # DISTANCE METRIC
            total_points_image += 1

            for alpha in CONFIGURATION["ALPHA"]:

                if dist <= alpha * max_dim:                     # PREDICTION IS CORRECT?
                    total_correct[alpha] += 1
                    total_correct_image[alpha] += 1

        # PRINT PER IMAGE

        for alpha in CONFIGURATION["ALPHA"]:
            total_correct_keypoints[alpha] += 100 * total_correct_image[alpha]
            total_correct_image[alpha] = round(100 * total_correct_image[alpha] / total_points_image, 2)
            total_correct[alpha] += total_correct_image[alpha]

        if visualize:
            visualize_keypoints(batch["src_path"][0], batch["trg_path"][0], batch["src_kps"][0],
                                pred_kps.cpu().numpy(),trg_kps.cpu().numpy())

    print("PCK@t results per image:")
    for alpha in CONFIGURATION["ALPHA"]:
        total_correct[alpha] = round(total_correct[alpha] / total_images, 2)
        total_correct_keypoints[alpha] = round(total_correct_keypoints[alpha] / total_keypoints, 2)
        print("PCK@" + str(alpha) + ": " + str(total_correct[alpha]) + "%")

    print()
    print("PCK@t results per keypoint:")
    for alpha in CONFIGURATION["ALPHA"]:
        print("PCK@" + str(alpha) + ": " + str(total_correct_keypoints[alpha]) + "%")

    return

# --- VISUALIZE RESULTS AND COMPARE CORRECT AND PREDICTED KEYPOINTS ON TARGET IMAGE. ---

def visualize_keypoints(src_path, trg_path, src_kps, pred_kps, trg_kps):
    src_img = cv2.imread(src_path)[:, :, ::-1]
    trg_img = cv2.imread(trg_path)[:, :, ::-1]

    (_, axes) = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(src_img)
    axes[0].scatter(src_kps[:,0], src_kps[:,1], c="r", s=40, label="src_kps")
    axes[0].set_title("Source Image")

    axes[1].imshow(trg_img)
    axes[1].scatter(pred_kps[:,0], pred_kps[:,1], c="b", s=40, label="pred_kps")
    axes[1].scatter(trg_kps[:,0], trg_kps[:,1], c="g", s=40, marker="X", label="gt_kps")
    axes[1].set_title("Target Image")

    plt.legend()
    plt.show()

In [ ]:
# @title Main

%pip install torchmetrics                # ONLY FOR FIRST EXECUTION

from google.colab import drive
# from Inference import run_evaluation
# from Train import Train_step
# from Configurations import CONFIGURATION, Loader
import torch, time

# --- IF SAM MODEL ---

if CONFIGURATION["MODEL_VERSION"] == "sam":
#     %pip install git+https://github.com/facebookresearch/segment-anything.git
    from segment_anything import sam_model_registry                 # DOWNLOAD

# --- MOUNT DRIVE AND EXTRACT DATASET ---

drive.mount(CONFIGURATION["PATH_DRIVE"], force_remount=True)
!tar -xzf {CONFIGURATION["PATH_FILE"]}

# --- LOAD MODEL ---

print()
print("Loading ", CONFIGURATION["MODEL_VERSION"], " model...")

if CONFIGURATION["MODEL_VERSION"] == "dinov2":
    MODEL = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14_reg", weights=CONFIGURATION["PTH_PATH_DINOV2"])
elif CONFIGURATION["MODEL_VERSION"] == "dinov3":
    MODEL = torch.hub.load("facebookresearch/dinov3", "dinov3_vitb16", weights=CONFIGURATION["PTH_PATH_DINOV3"])
elif CONFIGURATION["MODEL_VERSION"] == "sam":
    MODEL = sam_model_registry["vit_b"](checkpoint=CONFIGURATION["PTH_PATH_SAM"])
MODEL = MODEL.to(CONFIGURATION["DEVICE"])

# FINE TUNING PHASE

if CONFIGURATION["TUNING"]:
    Train_step()
    print("LOADING BEST MODEL FOUND FOR " + CONFIGURATION["MODEL_VERSION"])
    MODEL.load_state_dict(torch.load(CONFIGURATION["PATH_BEST_MODEL"]))            # BEST MODEL CHOSEN

# INFERENCE PHASE

if CONFIGURATION["DEVICE"] == "cuda":
    torch.cuda.synchronize()                         # SYNCHROIZE GPU

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)          # GO
    start_event.record()

else:
    start_event = time.time()       # GO

MODEL.eval()
loader = Loader(index=0)
run_evaluation(loader, "test")

if CONFIGURATION["DEVICE"] == "cuda":
    torch.cuda.synchronize()          # STOP
    end_event.record()
else:
    end_event = time.time()


elapsed_time = start_event.elapsed_time(end_event) / 1000              # SECONDS
print()
print("Analysis for: " + CONFIGURATION["DEVICE"])
print("Total required time: " + str(elapsed_time) + " seconds")

Mounted at /content/drive

Loading  dinov2  model...


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main



PERFORMING INITIAL EVALUATION ON PRE-TRAINED MODEL


Evaluating validation loss:   0%|          | 0/1070 [00:00<?, ?it/s]

Evaluating validation PCK metrics:   0%|          | 0/1070 [00:00<?, ?it/s]

...

Evaluating validation loss:   0%|          | 0/1070 [00:00<?, ?it/s]

Epoch train loss: 19.15519874446743
Epoch validation loss: 23.852873958382652



Evaluating validation PCK metrics:   0%|          | 0/1070 [00:00<?, ?it/s]

PCK@t results per image:
PCK@0.05: 55.14%
PCK@0.1: 70.85%
PCK@0.2: 83.63%

PCK@t results per keypoint:
PCK@0.05: 52.47%
PCK@0.1: 67.88%
PCK@0.2: 80.92%
BEST MODEL SAVED!

LOADING BEST MODEL FOUND FOR dinov2


Evaluating test PCK metrics:   0%|          | 0/2438 [00:00<?, ?it/s]